# Welcome to Colab!

In [ ]:
!pip install -qU sentence-transformers faiss-cpu pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3

In [ ]:
import json

# 1. Load các file chunks
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Giả sử bạn có 2 file chunk, hãy gộp chúng lại
chunks_1 = load_json('/content/nd168_chatbot_01.json')
chunks_2 = load_json('/content/nd336_chatbot_01.json')
all_chunks = chunks_1 + chunks_2

# Tạo list chứa text và list chứa ID tương ứng để map sau này
chunk_texts = [chunk['text_content'] for chunk in all_chunks]
chunk_ids = [chunk['chunk_id'] for chunk in all_chunks]

print(f"Tổng số chunks: {len(chunk_texts)}")

# 2. Load file ground truth
ground_truth = load_json('/content/ground_truth_dataset.json')
print(f"Tổng số câu hỏi đánh giá: {len(ground_truth)}")

Tổng số chunks: 1833
Tổng số câu hỏi đánh giá: 300


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load model BGE-M3
# Lưu ý: Lần đầu chạy sẽ mất chút thời gian tải model về
model = SentenceTransformer('BAAI/bge-m3')

# Encode tất cả các chunks thành vector
# normalize_embeddings=True giúp chuẩn hóa vector để dùng Dot Product (tương đương Cosine Similarity)
print("Đang embedding các chunks...")
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)

# Kiểm tra chiều của vector (BGE-M3 thường là 1024)
print(f"Kích thước ma trận embedding: {chunk_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Đang embedding các chunks...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

Kích thước ma trận embedding: (1833, 1024)


In [ ]:
import faiss

# Lấy số chiều của vector
dimension = chunk_embeddings.shape[1]

# Khởi tạo index FAISS (dùng Inner Product vì vector đã được normalize)
index = faiss.IndexFlatIP(dimension)

# Thêm vector của các chunks vào index
index.add(chunk_embeddings)

print(f"Đã thêm {index.ntotal} vectors vào FAISS Index.")

Đã thêm 1833 vectors vào FAISS Index.


KẾT QUẢ CHƯA RERANK

In [ ]:
# Cài đặt số lượng kết quả muốn lấy ra (ví dụ: lấy top 3 và top 5)
K_VALUES = [3, 5]

def evaluate_retrieval(k):
    hits = 0
    mrr_sum = 0

    for item in ground_truth:
        query = item['query']
        expected_ids = item['expected_chunk_ids']

        # 1. Embed query
        query_emb = model.encode([query], normalize_embeddings=True)

        # 2. Search trong FAISS
        distances, indices = index.search(query_emb, k)

        # 3. Lấy ra các chunk_id đã retrieve được
        retrieved_ids = [chunk_ids[idx] for idx in indices[0]]

        # 4. Kiểm tra xem có trúng không (Hit) và tính rank
        hit = False
        for rank, r_id in enumerate(retrieved_ids):
            if r_id in expected_ids:
                if not hit:
                    mrr_sum += 1.0 / (rank + 1)
                    hit = True

        if hit:
            hits += 1

    # Tính trung bình
    hit_rate = hits / len(ground_truth)
    mrr = mrr_sum / len(ground_truth)

    return hit_rate, mrr

# Chạy đánh giá
print("--- KẾT QUẢ ĐÁNH GIÁ RETRIEVAL BẰNG BGE-M3 ---")
for k in K_VALUES:
    hit_rate, mrr = evaluate_retrieval(k)
    print(f"Top-{k}:")
    print(f"  - Hit Rate@{k}: {hit_rate:.4f} ({hit_rate*100:.2f}%)")
    print(f"  - MRR@{k}     : {mrr:.4f}")
    print("-" * 30)

--- KẾT QUẢ ĐÁNH GIÁ RETRIEVAL BẰNG BGE-M3 ---
Top-3:
  - Hit Rate@3: 0.7000 (70.00%)
  - MRR@3     : 0.6239
------------------------------
Top-5:
  - Hit Rate@5: 0.7300 (73.00%)
  - MRR@5     : 0.6312
------------------------------


In [ ]:
!pip install -qU sentence-transformers faiss-cpu rank_bm25 numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 97.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompa

KẾT QUẢ KHI RERANK

In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# ==========================================
# 1. TẢI DỮ LIỆU & THÊM METADATA
# ==========================================
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Tải 2 file chunks (bạn sửa lại tên file cho đúng thực tế nhé)
chunks_1 = load_json('/content/nd168_chatbot_01.json')
chunks_2 = load_json('/content/nd336_chatbot_01.json')
all_chunks = chunks_1 + chunks_2
ground_truth = load_json('/content/ground_truth_dataset.json')

# Hàm ghép Metadata vào Text
def format_text_with_metadata(chunk):
    meta = chunk.get('metadata', {})
    chuong = meta.get('chuong', '')
    dieu = meta.get('dieu', '')
    khoan = meta.get('khoan', '')
    diem = meta.get('diem', '')

    meta_str = f"Chương {chuong}, Điều {dieu}"
    if khoan: meta_str += f", Khoản {khoan}"
    if diem: meta_str += f", Điểm {diem}"

    # Kết hợp: [Metadata] + Nội dung text
    return f"[{meta_str}] {chunk['text_content']}"

# Xử lý toàn bộ data
chunk_ids = [chunk['chunk_id'] for chunk in all_chunks]
chunk_texts_with_meta = [format_text_with_metadata(chunk) for chunk in all_chunks]

print(f"Tổng số chunks: {len(chunk_texts_with_meta)}")
print("Ví dụ Text có Metadata:", chunk_texts_with_meta[0])

# ==========================================
# 2. KHỞI TẠO CÁC MÔ HÌNH
# ==========================================
print("\nĐang tải các mô hình...")
# Mô hình Vector (Dense)
model = SentenceTransformer('BAAI/bge-m3')
# Mô hình Reranker (Cross-Encoder)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

# Khởi tạo BM25 (Sparse)
# Tách từ đơn giản bằng khoảng trắng (có thể dùng thu viện VNCoreNLP để tách từ tiếng Việt tốt hơn)
tokenized_corpus = [doc.lower().split(" ") for doc in chunk_texts_with_meta]
bm25 = BM25Okapi(tokenized_corpus)

# Nhúng Vector và tạo FAISS Index
print("\nĐang embedding vector...")
chunk_embeddings = model.encode(chunk_texts_with_meta, show_progress_bar=True, normalize_embeddings=True)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

# ==========================================
# 3. ĐÁNH GIÁ (HYBRID SEARCH -> RERANK)
# ==========================================
TOP_N_RETRIEVE = 20 # Số lượng kết quả lấy từ Vector và BM25 (trước khi gộp)
K_VALUES = [1, 3, 5]

def evaluate_ultimate_pipeline():
    results = {k: {"hits": 0, "mrr_sum": 0} for k in K_VALUES}

    print(f"\nĐang chạy đánh giá trên {len(ground_truth)} câu hỏi...")

    for item in ground_truth:
        query = item['query']
        expected_ids = item['expected_chunk_ids']

        # --- BƯỚC A: LẤY TOP N BẰNG BM25 ---
        tokenized_query = query.lower().split(" ")
        bm25_scores = bm25.get_scores(tokenized_query)
        # Lấy top N chỉ số có điểm cao nhất
        top_n_bm25_indices = np.argsort(bm25_scores)[::-1][:TOP_N_RETRIEVE]

        # --- BƯỚC B: LẤY TOP N BẰNG VECTOR (FAISS) ---
        query_emb = model.encode([query], normalize_embeddings=True)
        _, top_n_faiss_indices = index.search(query_emb, TOP_N_RETRIEVE)
        top_n_faiss_indices = top_n_faiss_indices[0]

        # --- BƯỚC C: KẾT HỢP (UNION) DANH SÁCH ---
        # Gộp danh sách index từ BM25 và FAISS, dùng 'set' để loại bỏ trùng lặp
        combined_indices = list(set(top_n_bm25_indices).union(set(top_n_faiss_indices)))

        # --- BƯỚC D: RERANK BẰNG CROSS-ENCODER ---
        # Lấy ra các ID và Text từ danh sách đã gộp
        candidate_ids = [chunk_ids[idx] for idx in combined_indices]
        candidate_texts = [chunk_texts_with_meta[idx] for idx in combined_indices]

        # Tạo cặp [Query, Text] để chấm điểm
        pairs = [[query, text] for text in candidate_texts]
        rerank_scores = reranker.predict(pairs)

        # Ghép ID và Điểm, sau đó sắp xếp giảm dần
        scored_candidates = list(zip(candidate_ids, rerank_scores))
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        final_ranked_ids = [x[0] for x in scored_candidates]

        # --- BƯỚC E: TÍNH TOÁN CHỈ SỐ ---
        for k in K_VALUES:
            top_k_ids = final_ranked_ids[:k]
            hit = False
            for rank, r_id in enumerate(top_k_ids):
                if r_id in expected_ids:
                    if not hit:
                        results[k]["mrr_sum"] += 1.0 / (rank + 1)
                        hit = True
            if hit:
                results[k]["hits"] += 1

    # In kết quả
    print("\n" + "="*50)
    print(" KẾT QUẢ: HYBRID SEARCH + METADATA + RERANKER")
    print("="*50)
    for k in K_VALUES:
        hit_rate = results[k]["hits"] / len(ground_truth)
        mrr = results[k]["mrr_sum"] / len(ground_truth)
        print(f"Top-{k}:")
        print(f"  - Hit Rate@{k}: {hit_rate:.4f} ({hit_rate*100:.2f}%)")
        print(f"  - MRR@{k}     : {mrr:.4f}")
        print("-" * 30)

# Kích hoạt chạy hàm
evaluate_ultimate_pipeline()

Tổng số chunks: 1833
Ví dụ Text có Metadata: [Chương I, Điều 1] Điều 1. Phạm vi điều chỉnh

Đang tải các mô hình...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]


Đang embedding vector...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]


Đang chạy đánh giá trên 300 câu hỏi...

 KẾT QUẢ: HYBRID SEARCH + METADATA + RERANKER
Top-1:
  - Hit Rate@1: 0.6733 (67.33%)
  - MRR@1     : 0.6733
------------------------------
Top-3:
  - Hit Rate@3: 0.7933 (79.33%)
  - MRR@3     : 0.7267
------------------------------
Top-5:
  - Hit Rate@5: 0.8200 (82.00%)
  - MRR@5     : 0.7327
------------------------------


In [ ]:
import os
import json
import pickle
import faiss
from google.colab import drive

# 1. Kết nối Google Drive và tạo thư mục lưu trữ
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/Vector_DB_Ultimate'
os.makedirs(save_path, exist_ok=True)
print(f"Thư mục lưu trữ: {save_path}")

# 2. Lưu FAISS Index (Vector)
faiss.write_index(index, f"{save_path}/faiss_bge_m3.index")

# 3. Lưu BM25 (Từ khóa) bằng thư viện pickle
with open(f"{save_path}/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25, f)

# 4. Lưu danh sách ID và Text đã gắn metadata
data_to_save = {
    "chunk_ids": chunk_ids,
    "chunk_texts_with_meta": chunk_texts_with_meta
}
with open(f"{save_path}/chunk_data.json", "w", encoding="utf-8") as f:
    json.dump(data_to_save, f, ensure_ascii=False)

print("✅ Đã lưu toàn bộ hệ thống (FAISS, BM25, Dữ liệu) lên Google Drive thành công!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục lưu trữ: /content/drive/MyDrive/Vector_DB_Ultimate
✅ Đã lưu toàn bộ hệ thống (FAISS, BM25, Dữ liệu) lên Google Drive thành công!


In [ ]:
import json
import pickle
import faiss
from google.colab import drive
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Kết nối lại Drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/Vector_DB_Ultimate'

print("Đang tải lại dữ liệu từ Drive...")

# 2. Tải FAISS Index
index = faiss.read_index(f"{save_path}/faiss_bge_m3.index")

# 3. Tải BM25 Model
with open(f"{save_path}/bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)

# 4. Tải danh sách ID và Text
with open(f"{save_path}/chunk_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    chunk_ids = data["chunk_ids"]
    chunk_texts_with_meta = data["chunk_texts_with_meta"]

print(f"✅ Đã tải xong! Hiện có {index.ntotal} chunks trong cơ sở dữ liệu.")

# 5. Khởi tạo lại 2 mô hình (Bắt buộc phải chạy vì Colab reset bộ nhớ RAM)
print("Đang tải lại model BGE-M3 và Reranker...")
model = SentenceTransformer('BAAI/bge-m3')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

print("🎉 Hệ thống đã sẵn sàng để truy vấn!")